# TITO outputs check

Reads the latest TITO cycle for one country from the public outputs endpoint, checks it against the expected layout, and draws the same maps as the forecast app: rainfall accumulation, max unit streamflow, soil saturation, flood inundation probability and impact-based forecasting.

Set `BASE_URL` and `COUNTRY` below, then run all cells.

In [ ]:
!pip -q install rasterio

In [ ]:
BASE_URL = "https://REPLACE.cloudfront.net/outputs"
COUNTRY = "guatemala"  # guatemala, haiti, barbados, antigua, comoros
STAT = "median"        # min, median, max
PERIOD = "forecast"    # nowcast, forecast

In [ ]:
import collections, io, re
import numpy as np, pandas as pd, requests, rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap

def get_json(path):
    r = requests.get(f"{BASE_URL}/{path}", timeout=60)
    r.raise_for_status()
    return r.json()

latest = get_json(f"{COUNTRY}/latest.json")
cycle = latest["cycle"]
index = get_json(f"{COUNTRY}/{cycle}/index.json")
paths = [f["path"] for f in index["files"]]
root = f"{BASE_URL}/{COUNTRY}/{cycle}"
print(f"cycle {cycle}, published {latest['published_utc']}")
print(f"{len(paths)} files, {sum(f['size'] for f in index['files'])/1e6:.0f} MB, image {latest['image_tag']}")

## Layout check

Expected per `<basin>_<resolution>/` folder: `scampr/ensOut1..10`, `stormlab/ensOutN_slM` (10 x 5), `stream_sat/ensOut1..10`, 18 rasters in `summary/`, and `fim/` and `ibf/` for pilot basins.

In [ ]:
EXPECT = {"scampr": 10, "stormlab": 50, "stream_sat": 10, "summary": 18}
MEMBER_FILES = ["maxq", "maxsm", "maxunitq", "qpeaccum", "qpfaccum", "results.json", "WA_"]

tree = collections.defaultdict(set)
for p in paths:
    parts = p.split("/")
    if len(parts) >= 3:
        tree[(parts[0], parts[1])].add(parts[2])

problems = []
for basin in sorted({b for b, _ in tree}):
    print(f"\n{basin}/")
    for folder in ["scampr", "stormlab", "stream_sat", "summary", "fim", "ibf", "warmup"]:
        items = tree.get((basin, folder), set())
        n = len(items)
        want = EXPECT.get(folder)
        flag = "" if want is None or n == want else f"  <-- expected {want}"
        if folder == "fim" and n == 0:
            flag = "  (none: only pilot basins have it)"
        if folder == "ibf" and n == 0:
            if tree.get((basin, "fim")):
                flag = "  <-- missing: basin has fim/ (fine only if the country has no IBF data)"
                problems.append(f"{basin}/ibf missing although fim/ exists")
            else:
                flag = "  (none: only pilot basins have it)"
        print(f"  {folder:<11} {n:>3}{flag}")
        if want is not None and n != want:
            problems.append(f"{basin}/{folder}: {n} != {want}")
    for folder in ["scampr", "stormlab", "stream_sat"]:
        for member in sorted(tree.get((basin, folder), set())):
            files = [p.split("/")[-1] for p in paths if p.startswith(f"{basin}/{folder}/{member}/")]
            missing = [m for m in MEMBER_FILES if not any(f.startswith(m) for f in files)]
            if missing:
                problems.append(f"{basin}/{folder}/{member} missing {missing}")

print("\nOK" if not problems else "\nPROBLEMS:\n  " + "\n  ".join(problems[:20]))

## Raster sanity

Opens every summary and flood raster over HTTP and checks the CRS, that it is not empty, and that values are in range.

In [ ]:
RANGES = {"maxsm": (0, 100), "prob_depth": (0, 1), "qpeaccum": (0, 2000), "maxunitq": (0, 500)}

from rasterio.io import MemoryFile

def fetch(path):
    r = requests.get(f"{root}/{path}", timeout=120)
    r.raise_for_status()
    return r.content

def read(path):
    with MemoryFile(fetch(path)) as mem, mem.open() as r:
        a = r.read(1, masked=True).astype(float)
        return a, r.crs, r.bounds

rows = []
for p in paths:
    if not p.endswith(".tif") or not ("/summary/" in p or "/combined_overbank/" in p):
        continue
    a, crs, _ = read(p)
    v = a.compressed()
    kind = next((k for k in RANGES if k in p.split("/")[-1]), None)
    lo, hi = RANGES.get(kind, (-np.inf, np.inf))
    ok = v.size > 0 and np.isfinite(v).all() and v.min() >= lo and v.max() <= hi
    rows.append(dict(file=p, crs=str(crs), cells=v.size, min=v.min() if v.size else None,
                     median=np.median(v) if v.size else None, max=v.max() if v.size else None, ok=ok))
checks = pd.DataFrame(rows)
display(checks)
print("all rasters OK" if checks.ok.all() else f"{(~checks.ok).sum()} raster(s) out of range")

## Forecast maps (app legends)

The three summary maps from the app, for the national grid. Change `STAT` and `PERIOD` above to see min or max, or the nowcast.

In [ ]:
LEGENDS = {
    "qpeaccum": ("Rainfall accumulation (mm)", [1, 5, 10, 25, 50, 75, 100, 150, 250, 1e5],
                 ["#d9f0a3", "#addd8e", "#41ab5d", "#ffffb2", "#fdae61", "#f46d43", "#d73027", "#a50026", "#6a3d9a"]),
    "maxunitq": ("Max unit streamflow (m3/s per km2)", [0.1, 1, 2, 4, 6, 10, 20, 1e5],
                 ["#bdbdbd", "#d9ef8b", "#fdae61", "#d73027", "#c51b7d", "#2c7bb6", "#08306b"]),
    "maxsm": ("Soil saturation (%)", [10, 30, 50, 70, 85, 95, 100.1],
              ["#f6e8c3", "#dfc27d", "#80cdc1", "#35978f", "#01665e", "#003c30"]),
}
basins = sorted({p.split("/")[0] for p in paths if "/summary/" in p})
national = max(basins, key=lambda b: int(re.search(r"(\d+)m$", b).group(1)))

def summary_path(basin, var):
    return next(p for p in paths if p.startswith(f"{basin}/summary/{var}_{PERIOD}_{STAT}."))

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, (var, (title, bounds, colors)) in zip(axes, LEGENDS.items()):
    a, _, b = read(summary_path(national, var))
    a = np.ma.masked_less(a, bounds[0])
    im = ax.imshow(a, extent=(b.left, b.right, b.bottom, b.top), interpolation="nearest",
                   cmap=ListedColormap(colors), norm=BoundaryNorm(bounds, len(colors)))
    ax.set_title(f"{title}\n{national} {PERIOD} {STAT}")
    ax.set_facecolor("#eef2f5")
    cb = fig.colorbar(im, ax=ax, shrink=0.7)
    cb.set_ticks(bounds[:-1])
plt.tight_layout()

### Interactive map

Rainfall accumulation on a basemap, to compare with the app. Pick another layer with `LAYER`.

In [ ]:
import folium
from matplotlib import cm

LAYER = "qpeaccum"
title, bounds, colors = LEGENDS[LAYER]
a, _, b = read(summary_path(national, LAYER))
norm = BoundaryNorm(bounds, len(colors))
rgba = ListedColormap(colors)(norm(a.filled(0)))
rgba[..., 3] = np.where(a.mask | (a.filled(0) < bounds[0]), 0, 0.8)
m = folium.Map(location=[(b.bottom + b.top) / 2, (b.left + b.right) / 2], zoom_start=7, tiles="OpenStreetMap")
folium.raster_layers.ImageOverlay(rgba, bounds=[[b.bottom, b.left], [b.top, b.right]], name=title).add_to(m)
folium.LayerControl().add_to(m)
m

## Flood inundation mapping

Probability that water depth reaches each threshold (overbank view), for each pilot site.

In [ ]:
from rasterio.warp import transform_bounds

sites = sorted({p.split("/")[3] for p in paths if "/fim/" in p and p.count("/") >= 4})
print("FIM sites:", sites or "none for this country")
BLUES = ListedColormap(["#c6dbef", "#6baed6", "#3f51d8", "#5b00b3"])
PBOUNDS = BoundaryNorm([0.05, 0.2, 0.5, 0.8, 1.01], 4)
for site in sites:
    probs = sorted((p for p in paths if f"/{site}/combined_overbank/prob_depth_ge_" in p),
                   key=lambda p: int(re.search(r"ge_(\d+)cm", p).group(1)))
    fig, axes = plt.subplots(1, len(probs), figsize=(5 * len(probs), 5))
    for ax, p in zip(np.atleast_1d(axes), probs):
        a, _, b = read(p)
        ax.imshow(np.ma.masked_less(a, 0.05), extent=(b.left, b.right, b.bottom, b.top),
                  cmap=BLUES, norm=PBOUNDS, interpolation="nearest")
        ax.set_title(f"{site}\nP(depth >= {re.search(r'ge_(\d+)cm', p).group(1)} cm)")
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    summary = get_json(f"{COUNTRY}/{cycle}/" + next(p for p in paths if p.endswith(f"{site}/pf_summary.json")))
    print(site, "trigger:", summary.get("trigger"))

### Flood probability on a basemap

In [ ]:
if sites:
    site = sites[0]
    p = next(p for p in paths if f"/{site}/combined_overbank/prob_depth_ge_10cm" in p)
    a, crs, b = read(p)
    w, s, e, n = transform_bounds(crs, "EPSG:4326", *b)
    rgba = BLUES(PBOUNDS(a.filled(0)))
    rgba[..., 3] = np.where(a.mask | (a.filled(0) < 0.05), 0, 0.85)
    m = folium.Map(location=[(s + n) / 2, (w + e) / 2], zoom_start=15, tiles="OpenStreetMap")
    folium.raster_layers.ImageOverlay(rgba, bounds=[[s, w], [n, e]], name="P(depth >= 10 cm)").add_to(m)
    display(m)

## Ensemble hydrographs

Discharge at each gauge from every member, grouped by rainfall source.

In [ ]:
ts = [p for p in paths if re.search(r"/ts\..+\.crest\..+\.csv$", p)]
fine = min(basins, key=lambda b: int(re.search(r"(\d+)m$", b).group(1)))
gauges = sorted({re.search(r"/ts\.(.+)\.crest\.", p).group(1) for p in ts if p.startswith(fine)})
COLORS = {"stream_sat": "tab:blue", "scampr": "tab:green", "stormlab": "tab:orange"}
fig, axes = plt.subplots(len(gauges), 1, figsize=(12, 3.5 * len(gauges)), squeeze=False)
for ax, g in zip(axes[:, 0], gauges):
    for p in (p for p in ts if p.startswith(fine) and f"/ts.{g}.crest." in p):
        src = p.split("/")[1]
        df = pd.read_csv(f"{root}/{p}", parse_dates=["Time"])
        ax.plot(df["Time"], df["Discharge(m^3 s^-1)"], color=COLORS.get(src, "grey"), alpha=0.35, lw=0.8)
    for src, c in COLORS.items():
        ax.plot([], [], color=c, label=src)
    ax.set_title(f"{fine} {g}"); ax.set_ylabel("m3/s"); ax.legend()
plt.tight_layout()

## Impact-based forecasting

Warning levels per receptor, when the country has IBF data.

In [ ]:
import geopandas as gpd

ibf = [p for p in paths if "/ibf/" in p]
print("\n".join(ibf) or "no ibf/ outputs in this cycle")
for p in (p for p in ibf if p.endswith(".json")):
    print(p, get_json(f"{COUNTRY}/{cycle}/{p}"))
for p in (p for p in ibf if p.endswith(".gpkg")):
    g = gpd.read_file(io.BytesIO(fetch(p)))
    print(p, len(g), "features; columns:", list(g.columns))
    col = next((c for c in g.columns if re.search(r"level|class|warning", c, re.I)), None)
    display(g.explore(column=col, tiles="OpenStreetMap") if col else g.head())